In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/lianapestereva/zoo-llm/lora_training_dataset1.xlsx


In [2]:
!pip install torch transformers peft datasets
!pip install accelerate pandas openpyxl bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.4 MB/s eta 0:00:00:00:0100:01


In [ ]:
import os
import pandas as pd
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from datasets import Dataset
import random
import numpy as np

df = pd.read_excel("/kaggle/input/datasets/lianapestereva/zoo-llm/lora_training_dataset1.xlsx")
df_adv = df[df["attack_type"] == "adversarial_input"].copy()
print(f"Найдено adversarial примеров: {len(df_adv)}")

def format_example(instruction):
    return f"<|gen_adversarial|> {instruction}"

texts = [format_example(row["instruction"]) for _, row in df_adv.iterrows()]
dataset = Dataset.from_dict({"text": texts})

train_test = dataset.train_test_split(test_size=0.2, seed=42)
val_test = train_test["test"].train_test_split(test_size=0.5, seed=42)  
train_dataset = train_test["train"]
val_dataset = val_test["train"]
test_dataset = val_test["test"]

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

model_name = "microsoft/phi-2"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token  
tokenizer.padding_side = "right"

from transformers import AutoConfig

config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
config.pad_token_id = tokenizer.pad_token_id 

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    config=config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False  

model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters() 

Найдено adversarial примеров: 109
Train: 87, Val: 11, Test: 11


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

trainable params: 3,932,160 || all params: 2,783,616,000 || trainable%: 0.1413


In [ ]:
def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt"
    )
    tokenized["labels"] = tokenized["input_ids"].clone()
    return tokenized

train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
test_dataset = test_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
training_args = TrainingArguments(
    output_dir="./lora_adversarial",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
    logging_steps=10,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    fp16=True,
    report_to="none",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

trainer.train()

model.save_pretrained("./lora_adversarial_final")
tokenizer.save_pretrained("./lora_adversarial_final")

from peft import PeftModel
from transformers import AutoConfig

config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
config.pad_token_id = tokenizer.pad_token_id

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    config=config,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model_adapter = PeftModel.from_pretrained(base_model, "./lora_adversarial_final")
model_adapter.eval()

generated_instructions = set()
num_to_generate = 1000
max_attempts = 2000
attempt = 0

prompt = "<|gen_adversarial|>"

while len(generated_instructions) < num_to_generate and attempt < max_attempts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model_adapter.device)
    with torch.no_grad():
        outputs = model_adapter.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.8,
            top_p=0.95,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if generated.startswith(prompt):
        generated = generated[len(prompt):].strip()
    generated = " ".join(generated.split())
    if len(generated) > 15 and generated not in generated_instructions:
        generated_instructions.add(generated)
    attempt += 1
    if attempt % 100 == 0:
        print(f"Сгенерировано {len(generated_instructions)} из {num_to_generate}")

adversarial_list = list(generated_instructions)

df_adversarial = pd.DataFrame({
    "id": range(1, len(adversarial_list)+1),
    "adversarial_prompt": adversarial_list,
    "attack_type": "adversarial_input",
    "source": "lora_generated"
})

df_adversarial.to_excel("generated_adversarial_dataset.xlsx", index=False)
print(f"Сгенерировано {len(df_adversarial)} уникальных adversarial запросов")

print("\nПримеры сгенерированных adversarial запросов:")
for i, row in df_adversarial.sample(10).iterrows():
    print(f"{row['id']}: {row['adversarial_prompt']}")

df_adversarial['length'] = df_adversarial['adversarial_prompt'].str.len()
print(f"\nСтатистика длины: min={df_adversarial['length'].min()}, "
      f"max={df_adversarial['length'].max()}, mean={df_adversarial['length'].mean():.1f}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss,Validation Loss
50,0.557344,0.706018
100,0.457275,0.652540


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]